In [7]:
# !pip install --no-cache-dir "numpy>=2.1.0"

In [8]:
# !pip install --no-cache-dir einops accelerate transformers huggingface-hub==0.28.0

In [9]:
# MOMENT sans ses dépendances (--no-deps) 
# Cela évite qu'il essaie de réinstaller NumPy 1.25.2
# !pip install --no-cache-dir momentfm --no-deps

In [6]:
"""
MOMENT Foundation Model — LSST Time Series Classification
"""

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from momentfm import MOMENTPipeline
import numpy as np
from sklearn.metrics import accuracy_score, classification_report
from tqdm import tqdm


# ─────────────────────────────────────────────────────────────
# Model
# ─────────────────────────────────────────────────────────────

class MOMENTClassifier(nn.Module):

    def __init__(
        self,
        n_classes: int = 14,
        freeze_encoder: bool = True,
        dropout: float = 0.3,
    ):
        super().__init__()

        # Load pretrained MOMENT
        self.moment = MOMENTPipeline.from_pretrained(
            "AutonLab/MOMENT-1-large",
            model_kwargs={
                "task_name": "classification",
                "n_channels": 1,
                "num_class": n_classes,
            },
        )

        self.moment.init()

        d_model = self.moment.config.d_model

        if freeze_encoder:
            for p in self.moment.parameters():
                p.requires_grad = False
            print("[INFO] MOMENT encoder frozen")
        else:
            print("[INFO] MOMENT encoder trainable")

        self.head = nn.Sequential(
            nn.Linear(d_model, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(dropout / 2),
            nn.Linear(256, n_classes),
        )

        self.n_classes = n_classes

    def forward(self, x):

        B, T, C = x.shape

        # Channel independent
        x_ci = x.permute(0,2,1)          # (B,C,T)
        x_ci = x_ci.reshape(B*C,1,T)     # (B*C,1,T)

        output = self.moment(
            x_enc=x_ci,
            input_mask=None
        )

        emb = output.embeddings

        if emb.dim() == 3:
            emb = emb.mean(dim=1)

        emb = emb.reshape(B, C, -1)
        emb = emb.mean(dim=1)

        logits = self.head(emb)

        return logits


# ─────────────────────────────────────────────────────────────
# Trainer
# ─────────────────────────────────────────────────────────────

class Trainer:

    def __init__(
        self,
        model,
        train_dl,
        val_dl,
        device="cuda" if torch.cuda.is_available() else "cpu",
        label_smoothing=0.1
    ):

        self.model = model.to(device)
        self.train_dl = train_dl
        self.val_dl = val_dl
        self.device = device

        self.criterion = nn.CrossEntropyLoss(
            label_smoothing=label_smoothing
        )

        self.history = {
            "train_loss": [],
            "val_loss": [],
            "val_acc": []
        }

    def _train_epoch(self, optimizer):

        self.model.train()

        total_loss = 0

        for x,y in tqdm(self.train_dl, desc="train", leave=False):

            x = x.to(self.device)
            y = y.to(self.device)

            optimizer.zero_grad()

            logits = self.model(x)

            loss = self.criterion(logits,y)

            loss.backward()

            nn.utils.clip_grad_norm_(
                self.model.parameters(),
                max_norm=1.0
            )

            optimizer.step()

            total_loss += loss.item()

        return total_loss / len(self.train_dl)


    @torch.no_grad()
    def _eval_epoch(self):

        self.model.eval()

        total_loss = 0
        all_preds = []
        all_targets = []

        for x,y in tqdm(self.val_dl, desc="val", leave=False):

            x = x.to(self.device)
            y = y.to(self.device)

            logits = self.model(x)

            loss = self.criterion(logits,y)

            total_loss += loss.item()

            preds = logits.argmax(dim=-1)

            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(y.cpu().numpy())

        val_loss = total_loss / len(self.val_dl)

        val_acc = accuracy_score(all_targets, all_preds)

        return val_loss, val_acc


    # Linear probing
    def linear_probe(self, epochs=10, lr=1e-3):

        print("\n===== PHASE 1: LINEAR PROBING =====")

        optimizer = AdamW(
            self.model.head.parameters(),
            lr=lr,
            weight_decay=1e-4
        )

        scheduler = CosineAnnealingLR(
            optimizer,
            T_max=epochs
        )

        best_acc = 0

        for epoch in range(1,epochs+1):

            train_loss = self._train_epoch(optimizer)

            val_loss, val_acc = self._eval_epoch()

            scheduler.step()

            print(
                f"Epoch {epoch}/{epochs} "
                f"train_loss={train_loss:.4f} "
                f"val_loss={val_loss:.4f} "
                f"val_acc={val_acc:.4f}"
            )

            if val_acc > best_acc:

                best_acc = val_acc

                torch.save(
                    self.model.state_dict(),
                    "best_moment_probe.pt"
                )

        print("Best probe acc:",best_acc)


    # Fine tuning
    def fine_tune(
        self,
        epochs=20,
        lr_backbone=1e-5,
        lr_head=1e-4
    ):

        print("\n===== PHASE 2: FINE TUNING =====")

        for p in self.model.moment.parameters():
            p.requires_grad = True

        optimizer = AdamW(
            [
                {
                    "params": self.model.moment.parameters(),
                    "lr": lr_backbone
                },
                {
                    "params": self.model.head.parameters(),
                    "lr": lr_head
                },
            ],
            weight_decay=1e-4
        )

        scheduler = CosineAnnealingLR(
            optimizer,
            T_max=epochs
        )

        best_acc = 0

        for epoch in range(1,epochs+1):

            train_loss = self._train_epoch(optimizer)

            val_loss, val_acc = self._eval_epoch()

            scheduler.step()

            print(
                f"Epoch {epoch}/{epochs} "
                f"train_loss={train_loss:.4f} "
                f"val_loss={val_loss:.4f} "
                f"val_acc={val_acc:.4f}"
            )

            if val_acc > best_acc:

                best_acc = val_acc

                torch.save(
                    self.model.state_dict(),
                    "best_moment_finetune.pt"
                )

        print("Best fine tune acc:",best_acc)


    @torch.no_grad()
    def evaluate_test(self, test_dl, checkpoint=None):

        if checkpoint:

            self.model.load_state_dict(
                torch.load(
                    checkpoint,
                    map_location=self.device
                )
            )

        self.model.eval()

        all_preds=[]
        all_targets=[]

        for x,y in tqdm(test_dl):

            x = x.to(self.device)

            logits = self.model(x)

            preds = logits.argmax(dim=-1)

            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(y.numpy())

        acc = accuracy_score(all_targets, all_preds)

        print("\nTEST ACCURACY:",acc)

        print(
            classification_report(
                all_targets,
                all_preds
            )
        )


# ─────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────

if __name__ == "__main__":

    from tslearn.datasets import UCR_UEA_datasets
    from dataloader import build_lsst_dataloader

    print("Loading LSST")

    ds = UCR_UEA_datasets()

    X_train, y_train, X_test, y_test = ds.load_dataset("LSST")

    train_dl, val_dl, test_dl = build_lsst_dataloader(
        X_train,
        y_train,
        X_test,
        y_test,
        window=None,
        stride=1,
        batch_size=32,
        num_workers=4
    )

    N_CLASSES = len(np.unique(y_train))

    model = MOMENTClassifier(
        n_classes=N_CLASSES,
        freeze_encoder=True,
        dropout=0.3
    )

    trainer = Trainer(
        model,
        train_dl,
        val_dl,
        label_smoothing=0.1
    )

    trainer.linear_probe(
        epochs=10,
        lr=1e-3
    )

    trainer.fine_tune(
        epochs=20,
        lr_backbone=1e-5,
        lr_head=1e-4
    )

    trainer.evaluate_test(
        test_dl,
        checkpoint="best_moment_finetune.pt"
    )

/opt/python/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading LSST


/opt/python/lib/python3.13/site-packages/momentfm/models/moment.py:174: UserWarning: Only reconstruction head is pre-trained. Classification and forecasting heads must be fine-tuned.
  warnings.warn("Only reconstruction head is pre-trained. Classification and forecasting heads must be fine-tuned.")


[INFO] MOMENT encoder frozen

===== PHASE 1: LINEAR PROBING =====


train:   0%|          | 0/61 [00:00<?, ?it/s]/opt/python/lib/python3.13/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/opt/python/lib/python3.13/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
                                                      

...

train:   0%|          | 0/61 [00:00<?, ?it/s]/opt/python/lib/python3.13/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
                                                        

Epoch 20/20 train_loss=0.9078 val_loss=2.5301 val_acc=0.3747
Best fine tune acc: 0.439918533604888


  0%|          | 0/78 [00:00<?, ?it/s]/opt/python/lib/python3.13/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
100%|██████████| 78/78 [17:57<00:00, 13.82s/it]


TEST ACCURACY: 0.5624493106244931
              precision    recall  f1-score   support

           0       0.35      0.30      0.32       124
           1       0.77      0.95      0.85       270
           2       0.38      0.45      0.41       382
           3       0.00      0.00      0.00        63
           4       1.00      0.14      0.25         7
           5       0.33      0.26      0.29        35
           6       0.12      0.05      0.07       153
           7       0.00      0.00      0.00        24
           8       0.62      0.90      0.74       313
           9       0.00      0.00      0.00        68
          10       0.85      0.87      0.86       121
          11       0.58      0.66      0.62       777
          12       0.00      0.00      0.00        77
          13       0.00      0.00      0.00        52

    accuracy                           0.56      2466
   macro avg       0.36      0.33      0.32      2466
weighted avg       0.48      0.56      0.51  


/opt/python/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/python/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/python/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
